In [37]:
# [CELL 1] STUDENT CONFIGURATION AND WORKSPACE
from pathlib import Path

LAST_NAME = "David"
STUDENT_ID = "TUPM-26-1110"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(exist_ok=True)

print("Student:", LAST_NAME, "| ID:", STUDENT_ID)
print("Seed:", SEED_DIGIT, "| ID Sum:", ID_SUM)
print("Workspace:", WORK_DIR.resolve())

Student: David | ID: TUPM-26-1110
Seed: 0 | ID Sum: 11
Workspace: C:\Users\Rachel\OneDrive - Technological University of the Philippines\DAVID\Lab6_File_Handling_David\Lab6_File_Handling_Additional_Exercises\Exercise2_Mobile_Device_Profiling


In [51]:
# [CELL 2] FILTER AND COMPUTE AVERAGES

import csv

dataset_path = DATA_DIR / "user_behavior_dataset.csv"

target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
target_class = (SEED_DIGIT % 5) + 1

total_records = 0
os_matches = 0
invalid = 0
selected = []

with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records += 1

        try:
            if row["Operating System"].strip() != target_os:
                continue

            os_matches += 1

            if int(row["User Behavior Class"]) != target_class:
                continue

            battery = float(row["Battery Drain (mAh/day)"])
            data_usage = float(row["Data Usage (MB/day)"])

        except (KeyError, ValueError, TypeError, AttributeError):
            invalid += 1
            continue

        selected.append({
            "User ID": row["User ID"].strip(),
            "Device Model": row["Device Model"].strip(),
            "Operating System": row["Operating System"].strip(),
            "Battery Drain (mAh/day)": battery,
            "Data Usage (MB/day)": data_usage,
        })

# COMPUTE AVERAGES
if selected:
    avg_battery = sum(
        r["Battery Drain (mAh/day)"] for r in selected
    ) / len(selected)

    avg_data = sum(
        r["Data Usage (MB/day)"] for r in selected
    ) / len(selected)
else:
    avg_battery = None
    avg_data = None

# DISPLAY RESULTS
print("Operating system:", target_os, "| Behavior class:", target_class)
print("Total records in dataset:", total_records)
print("Records matching operating system:", os_matches)
print("Records in target behavior class:", len(selected))
print("Invalid records skipped:", invalid)
print("Average battery drain (mAh/day):",
      round(avg_battery, 2) if avg_battery is not None else "No matching records")
print("Average data usage (MB/day):",
      round(avg_data, 2) if avg_data is not None else "No matching records")

Operating system: Android | Behavior class: 1
Total records in dataset: 700
Records matching operating system: 554
Records in target behavior class: 114
Invalid records skipped: 0
Average battery drain (mAh/day): 452.61
Average data usage (MB/day): 202.49


In [52]:
# [CELL 3] EXPORT, VERIFY COUNT, BACKUP
csv_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_profile.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_summary.json"

fields = ["User ID", "Device Model", "Operating System", "Battery Drain (mAh/day)", "Data Usage (MB/day)"]

with open(csv_out, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    writer.writerows(selected)

with open(csv_out, "r", encoding="utf-8", newline="") as file:
    exported_rows = sum(1 for _ in csv.DictReader(file))

count_verified = exported_rows == len(selected)

summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "operating_system": target_os,
    "user_behavior_class": target_class,
    "total_records": total_records,
    "records_matching_os": os_matches,
    "records_in_target_class": len(selected),
    "invalid_records": invalid,
    "average_battery_drain_mah_day": round(avg_battery, 2) if avg_battery is not None else None,
    "average_data_usage_mb_day": round(avg_data, 2) if avg_data is not None else None,
    "exported_row_count": exported_rows,
    "count_verified": count_verified,
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

backup_path = BACKUP_DIR / csv_out.name
shutil.copy2(csv_out, backup_path)

print("Count verification:", "VERIFIED" if count_verified else "MISMATCH", f"({exported_rows} exported / {len(selected)} selected)")
print("Exported CSV file:", csv_out)
print("JSON summary file:", json_path)
print("Backup file:", backup_path.name)
print("Backup exists:", backup_path.exists())
print("Backup size:", backup_path.stat().st_size, "bytes")

Count verification: VERIFIED (114 exported / 114 selected)
Exported CSV file: c:\Users\Rachel\OneDrive - Technological University of the Philippines\DAVID\Lab6_File_Handling_David\Lab6_File_Handling_Additional_Exercises\Exercise2_Mobile_Device_Profiling\output\DAVID_TUPM-26-1110_device_profile.csv
JSON summary file: c:\Users\Rachel\OneDrive - Technological University of the Philippines\DAVID\Lab6_File_Handling_David\Lab6_File_Handling_Additional_Exercises\Exercise2_Mobile_Device_Profiling\output\DAVID_TUPM-26-1110_device_summary.json
Backup file: DAVID_TUPM-26-1110_device_profile.csv
Backup exists: True
Backup size: 4547 bytes
